# kaiming-uniform-sf-init — ex2: Kaiming uniform SF init for Conv2d weights (fan_in = IC * kH * kW)

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `kaiming-uniform-sf-init`. Running the final beacon cell reports progress against the `Init: Kaiming uniform SF init` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    """Minimal Tensor wrapper for the ARENA-style manual-autograd drills.
    Wraps a raw `torch.Tensor` in `.array`. Carries optional `.recipe`,
    `.requires_grad`, and `.grad` (the accumulated gradient at leaves)."""
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
        self.grad = None
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Init: Kaiming uniform SF init` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`kaiming-uniform-sf-init`** (exercise 2). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "kaiming-uniform-sf-init"
DD_SUBTOPIC = "Init: Kaiming uniform SF init"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Kaiming uniform SF init for Conv2d weights — quick refresher

For Conv2d, `fan_in` is NOT `in_channels` alone — it's the size of the **receptive patch** that produces one output unit:

```
fan_in = in_channels * kernel_h * kernel_w
sf     = 1 / sqrt(fan_in)
weight ~ Uniform(-sf, +sf)              shape (OC, IC, kH, kW)
```

Same `Uniform(-sf, +sf)` recipe as `nn.Linear`; only the `fan_in` formula changes. PyTorch's `nn.Conv2d.reset_parameters` uses exactly this.

**Exemplar.** `IC=3, kH=3, kW=3 -> fan_in = 27, sf = 1/sqrt(27) ~= 0.192`. Weights of shape `(OC, 3, 3, 3)` are sampled on `(-0.192, 0.192)`. Empirical `std ~= sf / sqrt(3) ~= 0.111`.

### Exercise 2 — Kaiming uniform SF init for Conv2d weights (fan_in = IC * kH * kW)

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Apply
> LO: Apply the Conv2d-specific Kaiming-uniform recipe — `fan_in = in_channels * kernel_h * kernel_w`, `sf = 1/sqrt(fan_in)`, `weight ~ Uniform(-sf, +sf)` of shape `(out_channels, in_channels, kernel_h, kernel_w)`.
> Keywords: kaiming, uniform, init, conv2d, receptive-fan-in
> ```

**KCs targeted:** `kaiming-uniform-sf-init`, `conv-fan-in-formula`

Implement `kaiming_uniform_sf_conv2d(out_channels, in_channels, kernel_h, kernel_w, generator)`. The Conv2d weight initializer ARENA uses (and PyTorch's `nn.Conv2d` default):

1. `fan_in = in_channels * kernel_h * kernel_w` — NOT just `in_channels`. This is the size of the receptive PATCH that produces one output unit, not just the channel dimension.
2. `sf = 1 / sqrt(fan_in)`.
3. Sample `(out_channels, in_channels, kernel_h, kernel_w)` floats uniformly on `(-sf, +sf)` using `generator`.
4. Return as a `torch.Tensor` (4-D).

**Contrast with ex1.** Ex1's Linear init uses `fan_in = in_features`. For Conv2d the `fan_in` formula changes — the spatial extent of the kernel matters because every spatial position participates in the summation that produces one output activation. Same `Uniform(-sf, +sf)` recipe, different `fan_in`.

**Why this differs from `nn.Linear` viewed as 1x1 Conv.** A `Conv2d(IC, OC, kernel_size=1)` has `fan_in = IC * 1 * 1 = IC` — which matches `Linear(IC, OC)`. The two coincide at kernel size 1. They diverge for bigger kernels: a 3x3 conv has `fan_in = 9 * IC` — `sf` shrinks by `sqrt(9) = 3`.

Hint: `t.rand(shape, generator=g)` is uniform on `[0, 1)`. To get `(-sf, +sf)`, do `(t.rand(shape, generator=g) * 2 - 1) * sf`.

Output: `torch.Tensor` of shape `(out_channels, in_channels, kernel_h, kernel_w)`.

In [ ]:
def kaiming_uniform_sf_conv2d(
    out_channels: int, in_channels: int, kernel_h: int, kernel_w: int,
    generator: t.Generator,
) -> Tensor:
    fan_in = in_channels * kernel_h * kernel_w
    sf = fan_in ** -0.5
    raw = t.rand(out_channels, in_channels, kernel_h, kernel_w, generator=generator)
    return (raw * 2 - 1) * sf


<details><summary>Solution</summary>

```python
def kaiming_uniform_sf_conv2d(
    out_channels: int, in_channels: int, kernel_h: int, kernel_w: int,
    generator: t.Generator,
) -> Tensor:
    fan_in = in_channels * kernel_h * kernel_w
    sf = fan_in ** -0.5
    raw = t.rand(out_channels, in_channels, kernel_h, kernel_w, generator=generator)
    return (raw * 2 - 1) * sf
```

**Why `fan_in = IC * kH * kW`, not `IC`.** For a forward Conv2d, each output activation is the sum of `IC * kH * kW` weighted inputs. The Kaiming derivation argues that the input-output variance is preserved when `Var(w) * fan_in = 1` (for ReLU it's 2; for the SF form, it's whatever the constant works out to under `Uniform(-sf, sf)`). The relevant `fan_in` is the count of inputs AGGREGATED per output unit. For Conv2d that's `IC * kH * kW` — every spatial position of the kernel patch.

**At kernel 1x1, Conv2d == Linear.** `fan_in = IC * 1 * 1 = IC`. Both inits sample on the same `(-1/sqrt(IC), +1/sqrt(IC))` interval. This is why 1x1 convs are sometimes called 'pointwise linear layers' — they really are linear in the channel dimension, with no spatial context.

**Why bigger kernels → smaller weights.** A 3x3 conv at the same `IC` has `9x` more inputs feeding each output, so each individual weight should be `3x` smaller on average to preserve the pre-activation scale. The test asserts the 1x1 vs 3x3 ratio is `~1/3 = sqrt(1/9)`.

**Why this matters at training time.** Wrong `fan_in` (using `IC` alone for a `3x3` conv) gives weights `3x` too large — pre-activations explode by `~3x` per layer, saturating ReLUs and killing gradients in the first few backward passes. The empirical-std test catches this.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex2',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()